# 🛠️ Aula 17 — Manutenção Preditiva e RUL

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** `trocador_degradacao.csv` — 12 meses de degradação do U de um trocador

---

## Contexto

Você é o engenheiro de confiabilidade da planta. O trocador E-101 apresenta queda progressiva de desempenho (incrustação). Seu chefe quer saber: **quando trocar?**

| Abordagem | Descrição |
|-----------|-----------|
| Corretiva | Quebrou → para a planta |
| Preventiva | Troca por calendário (mesmo se bom) |
| **Preditiva** | Troca quando os dados indicam falha próxima |

## RUL (Remaining Useful Life)

É o **tempo restante até cruzar o threshold de falha**:

$$\text{Curva: } U(t) = U_0 e^{-t/\tau} \qquad \text{Threshold: } U_{th} = 0.6 \cdot U_0$$

$$\text{RUL}(t) = -\tau \ln\left(\frac{U_{th}}{U_0}\right) - t$$

## 3.1 — Exercício Guiado: RUL com Curva + ML

Ajuste a curva exponencial E treine XGBoost para predizer o RUL. Compare.

### Passo 1: Carregar dados e visualizar

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula17/trocador_degradacao.csv"
df = pd.read_csv(URL)
print(df.head())
print(df.describe().round(1))

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df['mes'], df['U_W_m2K'], 'o-', color='steelblue', linewidth=2)
U0 = df['U_W_m2K'].iloc[0]
U_th = 0.6 * U0
plt.axhline(U_th, color='red', ls='--', label=f'Threshold (60% U0 = {U_th:.0f})')
plt.xlabel('Mês'); plt.ylabel('U (W/m²K)')
plt.title('Degradação do trocador E-101 (incrustação)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Passo 2: Ajustar curva exponencial

In [ ]:
def exp_modelo(t, u0, tau):
    return u0 * np.exp(-t / tau)

popt, _ = curve_fit(exp_modelo, df['mes'], df['U_W_m2K'], p0=[500, 20])
u0_fit, tau_fit = popt
print(f"Curva ajustada: U0={u0_fit:.1f}, tau={tau_fit:.1f} meses")

### Passo 3: RUL exponencial

Calcule o tempo t* em que U cruza o threshold e o RUL para cada mês.

In [ ]:
U_th = 0.6 * u0_fit
t_star = -tau_fit * np.log(U_th / u0_fit)
rul_exp = t_star - df['mes']
print(f"Threshold: {U_th:.1f} | Tempo da falha t*: {t_star:.1f} meses")
print(f"RUL exponencial no mês 6: {rul_exp.iloc[5]:.1f} meses")
print(f"RUL exponencial no mês 9: {rul_exp.iloc[8]:.1f} meses")

### Passo 4: RUL com XGBoost (ML direto)

Treine XGBoost para predizer o RUL a partir das features de operação.

In [ ]:
# Label: RUL verdadeiro (tempo restante até cruzar threshold — usando tau efetivo
# que depende das features de operação: mais vazão/ciclos = degradação mais rápida)
features = ['mes', 'vazao_media_L_min', 'T_entrada_media_C', 'ciclos_termicos']
X = df[features]

# RUL verdadeiro calculado da física com parâmetros da SG
# (Na prática, viemos da Aula 17: usamos a curva exponencial ajustada como label)
y_rul = rul_exp

split = int(len(X) * 0.8)
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0)
xgb.fit(X.iloc[:split], y_rul.iloc[:split])
pred = xgb.predict(X.iloc[split:])
mae_xgb = mean_absolute_error(y_rul.iloc[split:], pred)
mae_exp = mean_absolute_error(y_rul.iloc[split:], rul_exp.iloc[split:])
print(f"MAE (meses) — exponencial: {mae_exp:.2f} | XGBoost: {mae_xgb:.2f}")
print(f"XGBoost RUL no mês 6: {xgb.predict(X.iloc[5:6])[0]:.1f} meses")

### Passo 5: Plotar RUL ao longo do tempo

In [ ]:
xgb_rul = xgb.predict(X)
plt.figure(figsize=(10, 4))
plt.plot(df['mes'], rul_exp, 'o-', color='orange', label='RUL exponencial')
plt.plot(df['mes'], xgb_rul, 's--', color='green', label='RUL XGBoost')
plt.axhline(0, color='red', ls='--', label='Falha')
plt.xlabel('Mês'); plt.ylabel('RUL (meses)')
plt.title('RUL ao longo do tempo')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### ✏️ Pausa reflexiva (2 min)

O $R^2$ do modelo de RUL é bom? Se não, que feature está faltando?

> _Escreva aqui..._

## 3.2 — Exercício em Grupo: Threshold e Decisão

Cada grupo testa um threshold diferente e calcula o custo total esperado.

**Custos:** troca programada = R$ 50k | falha não-programada = R$ 500k

| Grupo | Threshold |
|-------|-----------|
| **A** | 50% de U0 |
| **B** | 60% de U0 |
| **C** | 70% de U0 |
| **D** | 80% de U0 |

In [ ]:
threshold_frac = 0.60   # ← mude para o do seu grupo (0.5, 0.6, 0.7, 0.8)
custo_troca = 50_000    # R$
custo_falha = 500_000   # R$

# Quando o threshold seria cruzado (RUL = 0)?
U_th_g = threshold_frac * u0_fit
t_falha_g = -tau_fit * np.log(U_th_g / u0_fit)
mes_troca = int(np.ceil(t_falha_g))

# Se trocarmos 1 mês antes da falha => custo de troca. Antes disso, risco de falha.
print(f"Threshold {threshold_frac*100:.0f}% de U0: falha em ~{t_falha_g:.1f} meses")
print(f"Custo esperado (trocar 1 mês antes): R$ {custo_troca:,.0f}")
print(f"Custo de falha não-programada: R$ {custo_falha:,.0f}")

### 🧠 Desafio extra (NT)

Se a incrustação acelera com o tempo (não-linear), um modelo linear de degradação subestima ou superestima a RUL? Qual o risco financeiro de cada erro?

> _Escreva aqui..._

---

## Checklist

- [ ] Threshold de falha definido
- [ ] Curva exponencial ajustada
- [ ] RUL histórico calculado
- [ ] Modelo XGBoost treinado
- [ ] MAE comparado
- [ ] Incerteza/intervalo de confiança
- [ ] Recomendação de troca